In [ ]:
# Needed Packages
#pip install transformers sentencepiece torch
import os

In [ ]:
os.chdir("C:/Users/spike/Desktop/CS 329 Studying and Tests")

In [ ]:
from Dialect_Detector import detect_dialect

In [ ]:
# Load the OPUS-MT Arabic→English Model

from transformers import MarianMTModel, MarianTokenizer
import torch

MODEL_NAME = "Helsinki-NLP/opus-mt-ar-en"

tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)




In [ ]:
# Dialect Normalization Layer (Key Innovation)
# Simple normalization dictionaries per dialect
DIALECT_NORMALIZATION = {
    "egyptian": {
        "رايح": "ذاهب",
        "عايز": "أريد",
        "مش": "ليس",
        "فين": "أين",
        "ليه": "لماذا"
    },
    "levantine": {
        "راح": "ذهب",
        "بدّي": "أريد",
        "مو": "ليس",
        "وين": "أين"
    },
    "gulf": {
        "وين": "أين",
        "أبغى": "أريد",
        "مو": "ليس"
    }
}

def normalize_dialect(text: str, dialect: str):
    """
    Replace dialectal words with MSA equivalents.
    Returns normalized text + list of applied rules.
    """
    applied_rules = []

    rules = DIALECT_NORMALIZATION.get(dialect, {})
    tokens = text.split()

    normalized_tokens = []
    for token in tokens:
        if token in rules:
            normalized_tokens.append(rules[token])
            applied_rules.append(f"{token} → {rules[token]}")
        else:
            normalized_tokens.append(token)

    normalized_text = " ".join(normalized_tokens)
    return normalized_text, applied_rules


In [ ]:
# Translation Function (OPUS-MT Inference)

def translate_ar_to_en(text: str, max_length: int = 128):
    """
    Translate Arabic text to English using OPUS-MT.
    """
    inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        translated = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=5,
            early_stopping=True
        )

    return tokenizer.decode(translated[0], skip_special_tokens=True)


In [ ]:
# Explanation Layer (Ambiguity Awareness)
# This is post-processing, not modeling

AMBIGUOUS_WORDS = {
    "راح": ["went", "will go"],
    "كان": ["was", "used to be"],
    "طلع": ["went out", "turned out", "appeared"]
}

def get_ambiguity_notes(text: str):
    notes = []
    for word, meanings in AMBIGUOUS_WORDS.items():
        if word in text:
            notes.append({
                "word": word,
                "possible_meanings": meanings
            })
    return notes


In [ ]:
# Main Pipeline (This Is What You Demo)
# Assumes You already have detect_dialect(text) implemented.


def dialect_aware_translate(text: str, detect_dialect):
    """
    Full pipeline:
    dialect detection → normalization → translation → explanations
    """
    dialect = detect_dialect(text)

    normalized_text, applied_rules = normalize_dialect(text, dialect)

    translation = translate_ar_to_en(normalized_text)

    ambiguity_notes = get_ambiguity_notes(text)

    return {
        "input_text": text,
        "detected_dialect": dialect,
        "normalized_text": normalized_text,
        "applied_normalization_rules": applied_rules,
        "translation": translation,
        "ambiguities": ambiguity_notes
    }


In [ ]:
# Example Run (Perfect for Live Demo)

example = "أنا رايح البيت ومش تعبان"

result = dialect_aware_translate(example, detect_dialect_dummy)

for k, v in result.items():
    print(f"{k}: {v}")
